# 01 — RV Variability Labels

Assign binary/single star labels based on radial-velocity scatter (VSCATTER) thresholds from the APOGEE DR17 allStar catalog. Stars with high visit-to-visit RV variability are candidate spectroscopic binaries; stars with low scatter are presumed single.

In [ ]:
import os
import sys

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from astropy.table import Table

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from config import LABEL_CONFIG, VIS_CONFIG
from src.data import load_allstar_catalog, filter_allstar_catalog
from src.utils import crossmatch_catalogs

%matplotlib inline
plt.rcParams["figure.dpi"] = VIS_CONFIG["dpi"]

## 1. Load Filtered Catalog

In [ ]:
catalog = Table.read("../../data/allstar_filtered.fits")
print(f"Filtered catalog: {len(catalog)} stars, {len(catalog.colnames)} columns")
print(f"Columns: {catalog.colnames[:15]}...")
print(f"VSCATTER range: [{catalog['VSCATTER'].min():.3f}, {catalog['VSCATTER'].max():.3f}] km/s")

## 2. Assign Labels

Labeling strategy based on APOGEE VSCATTER (standard deviation of visit-to-visit radial velocities):

- **Binary** (`label = 1`): VSCATTER > 1.0 km/s --- large RV variability consistent with orbital motion
- **Single** (`label = 0`): VSCATTER < 0.3 km/s --- low scatter consistent with measurement noise for a single star
- **Discarded**: 0.3 <= VSCATTER <= 1.0 km/s --- ambiguous intermediate zone, excluded from training

In [ ]:
binary_thresh = LABEL_CONFIG["rv_scatter_binary_threshold"]
single_thresh = LABEL_CONFIG["rv_scatter_single_threshold"]

binary_mask = catalog["VSCATTER"] > binary_thresh
single_mask = catalog["VSCATTER"] < single_thresh
intermediate_mask = ~binary_mask & ~single_mask

print(f"Thresholds: binary > {binary_thresh} km/s, single < {single_thresh} km/s")
print(f"  Binary candidates:  {binary_mask.sum():>7d}  ({binary_mask.sum()/len(catalog)*100:.1f}%)")
print(f"  Single candidates:  {single_mask.sum():>7d}  ({single_mask.sum()/len(catalog)*100:.1f}%)")
print(f"  Intermediate (drop): {intermediate_mask.sum():>6d}  ({intermediate_mask.sum()/len(catalog)*100:.1f}%)")

# Build labeled subset
labeled_mask = binary_mask | single_mask
labeled = catalog[labeled_mask].copy()
labeled["label"] = np.where(
    labeled["VSCATTER"] > binary_thresh, 1, 0
).astype(np.int16)

n_binary = (labeled["label"] == 1).sum()
n_single = (labeled["label"] == 0).sum()
print(f"\nLabeled sample: {len(labeled)} stars")
print(f"  Binary (label=1): {n_binary}  ({n_binary/len(labeled)*100:.1f}%)")
print(f"  Single (label=0): {n_single}  ({n_single/len(labeled)*100:.1f}%)")
print(f"  Class ratio (single:binary): {n_single/n_binary:.1f}:1")

## 3. Check for Systematic Biases

Verify that the VSCATTER-based labels are not systematically driven by stellar parameters (Teff, logg, SNR, metallicity). If binary-labeled stars cluster in a specific region of parameter space, the classifier may learn stellar type rather than true binarity.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

params = [
    ("TEFF", "Teff (K)"),
    ("LOGG", "log g"),
    ("SNR", "Combined S/N"),
    ("FE_H", "[Fe/H]"),
]

colors = {0: "steelblue", 1: "firebrick"}
labels_map = {0: "Single", 1: "Binary"}

for ax, (col, xlabel) in zip(axes.flat, params):
    for lbl in [0, 1]:
        mask = labeled["label"] == lbl
        ax.scatter(
            labeled[col][mask], labeled["VSCATTER"][mask],
            s=1, alpha=0.15, c=colors[lbl], label=labels_map[lbl],
            rasterized=True,
        )
    ax.set_xlabel(xlabel)
    ax.set_ylabel("VSCATTER (km/s)")
    ax.set_yscale("log")
    ax.axhline(binary_thresh, color="firebrick", ls="--", lw=0.8, alpha=0.5)
    ax.axhline(single_thresh, color="steelblue", ls="--", lw=0.8, alpha=0.5)
    ax.legend(markerscale=5, fontsize=8)

fig.suptitle("VSCATTER vs Stellar Parameters by Label", fontsize=14)
plt.tight_layout()

os.makedirs("../../figures", exist_ok=True)
fig.savefig("../../figures/label_systematics.png", dpi=VIS_CONFIG["dpi"], bbox_inches="tight")
print("Saved figures/label_systematics.png")

In [ ]:
# Compare median stellar parameters between binary and single classes
print("Median stellar parameters by class:")
print(f"{'Parameter':<12} {'Single':>10} {'Binary':>10} {'Difference':>12}")
print("-" * 48)

for col, name in [("TEFF", "Teff"), ("LOGG", "logg"), ("SNR", "SNR"), ("FE_H", "[Fe/H]"), ("VSCATTER", "VSCATTER")]:
    med_single = np.median(labeled[col][labeled["label"] == 0])
    med_binary = np.median(labeled[col][labeled["label"] == 1])
    print(f"{name:<12} {med_single:>10.3f} {med_binary:>10.3f} {med_binary - med_single:>+12.3f}")

## 4. Save Labeled Dataset

In [ ]:
out_path = "../../data/labeled_sample.fits"
labeled.write(out_path, overwrite=True)

n_binary = (labeled["label"] == 1).sum()
n_single = (labeled["label"] == 0).sum()

print(f"Saved labeled catalog to {out_path}")
print(f"  Total labeled stars: {len(labeled)}")
print(f"  Binary (label=1):    {n_binary}")
print(f"  Single (label=0):    {n_single}")
print(f"  Class ratio (single:binary): {n_single/n_binary:.1f}:1")